# 13. Perbandingan Stacking Sebelum dan Sesudah Calibration

Notebook ini membandingkan hasil Stacking **tanpa calibration** dan **dengan Platt Scaling** menggunakan probabilitas OOF yang sudah tersimpan di folder `artifacts`. Tidak ada model yang dilatih ulang.

**Notebook ini hanya membaca file dan tidak menulis apa pun**, sehingga aman dijalankan berulang kali dan tidak mengubah isi `artifacts`.

File yang dibaca:
- `data/final_split/train_final.csv` (hanya kolom target)
- `artifacts/stacking_oof_proba_uncalibrated.npy`
- `artifacts/stacking_oof_proba_calibrated.npy`
- `artifacts/prediabetes_threshold.txt`
- `artifacts/prediabetes_threshold_calibrated.txt`

## 0. Mengatur Folder Kerja

Notebook ini disimpan di folder `notebooks`, sedangkan data dan artifacts berada di folder `glysense_backend`. Sel berikut memindahkan folder kerja satu tingkat ke atas. Aman dijalankan berulang.

In [1]:
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print(os.getcwd())

d:\Tugas Akhir\Aplikasi Diabetes\glysense_backend


## 1. Membaca Data dan Probabilitas Tersimpan

In [2]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    recall_score,
    f1_score,
    accuracy_score,
    log_loss,
    confusion_matrix
)

y = pd.read_csv(
    "data/final_split/train_final.csv",
    usecols=["Diabetes_012"]
)["Diabetes_012"].astype(int).to_numpy()

proba_uncal = np.load("artifacts/stacking_oof_proba_uncalibrated.npy")
proba_cal = np.load("artifacts/stacking_oof_proba_calibrated.npy")

with open("artifacts/prediabetes_threshold.txt") as f:
    thr_uncal = float(f.read().strip())

with open("artifacts/prediabetes_threshold_calibrated.txt") as f:
    thr_cal = float(f.read().strip())

print("Jumlah target          :", len(y))
print("Bentuk proba uncal     :", proba_uncal.shape)
print("Bentuk proba cal       :", proba_cal.shape)
print("Threshold uncalibrated :", thr_uncal)
print("Threshold calibrated   :", thr_cal)

# Pengecekan konsistensi
assert len(y) == len(proba_uncal) == len(proba_cal), "Jumlah baris tidak sama!"
print("\nSetiap baris probabilitas berjumlah 1 (uncal):", np.allclose(proba_uncal.sum(axis=1), 1))
print("Setiap baris probabilitas berjumlah 1 (cal)  :", np.allclose(proba_cal.sum(axis=1), 1))

Jumlah target          : 202944
Bentuk proba uncal     : (202944, 3)
Bentuk proba cal       : (202944, 3)
Threshold uncalibrated : 0.39
Threshold calibrated   : 0.03

Setiap baris probabilitas berjumlah 1 (uncal): True
Setiap baris probabilitas berjumlah 1 (cal)  : True


## 2. Fungsi Bantu

`predict_with_threshold` memakai aturan yang sama dengan script threshold tuning dan `predict.py`: jika probabilitas Prediabetes melewati threshold maka kelas 1, jika tidak pilih antara kelas 0 dan 2 berdasarkan probabilitas terbesar.

In [3]:
def predict_with_threshold(proba, threshold):
    return np.where(
        proba[:, 1] >= threshold,
        1,
        np.where(proba[:, 0] >= proba[:, 2], 0, 2)
    )


def calculate_metrics(y_true, y_pred):
    recalls = recall_score(
        y_true, y_pred, labels=[0, 1, 2], average=None, zero_division=0
    )
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    accuracy = accuracy_score(y_true, y_pred)
    gmean = np.prod(recalls) ** (1 / len(recalls))
    return recalls, macro_f1, gmean, accuracy


def metrics_row(name, y_true, y_pred):
    recalls, macro_f1, gmean, accuracy = calculate_metrics(y_true, y_pred)
    return {
        "Skenario": name,
        "Recall_0": recalls[0],
        "Recall_1": recalls[1],
        "Recall_2": recalls[2],
        "Macro_F1": macro_f1,
        "G_Mean": gmean,
        "Accuracy": accuracy
    }

## 3. Perbandingan dengan Argmax

Prediksi diambil dari kelas dengan probabilitas tertinggi, tanpa threshold khusus.

In [4]:
rows = [
    metrics_row("Uncalibrated (argmax)", y, np.argmax(proba_uncal, axis=1)),
    metrics_row("Calibrated (argmax)", y, np.argmax(proba_cal, axis=1)),
]

print(pd.DataFrame(rows).round(4).to_string(index=False))

             Skenario  Recall_0  Recall_1  Recall_2  Macro_F1  G_Mean  Accuracy
Uncalibrated (argmax)    0.6138    0.3679    0.5976    0.4203  0.5129    0.6071
  Calibrated (argmax)    0.9832    0.0000    0.1514    0.3858  0.0000    0.8494


## 4. Perbandingan pada Threshold yang Tersimpan

Memakai threshold yang sudah dicari sebelumnya untuk masing-masing skenario. Ini adalah angka yang sebanding dengan hasil yang Anda lihat pada script threshold tuning.

In [5]:
pred_uncal_thr = predict_with_threshold(proba_uncal, thr_uncal)
pred_cal_thr = predict_with_threshold(proba_cal, thr_cal)

rows = [
    metrics_row(f"Uncalibrated + threshold {thr_uncal:.2f}", y, pred_uncal_thr),
    metrics_row(f"Calibrated + threshold {thr_cal:.2f}", y, pred_cal_thr),
]

print(pd.DataFrame(rows).round(4).to_string(index=False))

print("\nConfusion Matrix - Uncalibrated + threshold:")
print(confusion_matrix(y, pred_uncal_thr))

print("\nConfusion Matrix - Calibrated + threshold:")
print(confusion_matrix(y, pred_cal_thr))

                     Skenario  Recall_0  Recall_1  Recall_2  Macro_F1  G_Mean  Accuracy
Uncalibrated + threshold 0.39    0.6303    0.4251    0.5205    0.4158  0.5186    0.6112
  Calibrated + threshold 0.03    0.8297    0.3552    0.1455    0.3856  0.3500    0.7257

Confusion Matrix - Uncalibrated + threshold:
[[107751  38820  24391]
 [   905   1575   1225]
 [  4047   9513  14717]]

Confusion Matrix - Calibrated + threshold:
[[141845  26368   2749]
 [  2162   1316    227]
 [ 15448   8714   4115]]


## 5. Pencarian Ulang Threshold (Hanya di Memori)

Pengecekan ini memakai kriteria yang sama dengan script asli (G-Mean, lalu Recall Prediabetes, lalu Macro F1). Hasilnya **tidak disimpan**.

Rentang pencarian mengikuti script asli: 0.05 sampai 0.60 untuk uncalibrated, dan 0.01 sampai 0.40 untuk calibrated. Jika threshold terbaik berada di ujung rentang, kemungkinan nilai yang lebih baik ada di luar rentang.

In [6]:
def search_threshold(proba, thresholds):
    best = None
    best_score = (-1, -1, -1)

    for threshold in thresholds:
        pred = predict_with_threshold(proba, threshold)
        recalls, macro_f1, gmean, accuracy = calculate_metrics(y, pred)
        score = (gmean, recalls[1], macro_f1)

        if score > best_score:
            best_score = score
            best = threshold

    return best


grid_uncal = np.arange(0.05, 0.61, 0.01)
grid_cal = np.arange(0.01, 0.41, 0.01)

best_uncal = search_threshold(proba_uncal, grid_uncal)
best_cal = search_threshold(proba_cal, grid_cal)

print(f"Threshold terbaik uncalibrated : {best_uncal:.2f} (rentang {grid_uncal[0]:.2f}-{grid_uncal[-1]:.2f})")
print(f"Threshold terbaik calibrated   : {best_cal:.2f} (rentang {grid_cal[0]:.2f}-{grid_cal[-1]:.2f})")

if np.isclose(best_uncal, grid_uncal[0]) or np.isclose(best_uncal, grid_uncal[-1]):
    print("PERHATIAN: threshold uncalibrated berada di ujung rentang pencarian.")

if np.isclose(best_cal, grid_cal[0]) or np.isclose(best_cal, grid_cal[-1]):
    print("PERHATIAN: threshold calibrated berada di ujung rentang pencarian.")

print("\nSama dengan threshold tersimpan?")
print("Uncalibrated:", np.isclose(best_uncal, thr_uncal))
print("Calibrated  :", np.isclose(best_cal, thr_cal))

rows = [
    metrics_row(f"Uncalibrated + best {best_uncal:.2f}", y, predict_with_threshold(proba_uncal, best_uncal)),
    metrics_row(f"Calibrated + best {best_cal:.2f}", y, predict_with_threshold(proba_cal, best_cal)),
]

print()
print(pd.DataFrame(rows).round(4).to_string(index=False))

Threshold terbaik uncalibrated : 0.39 (rentang 0.05-0.60)
Threshold terbaik calibrated   : 0.03 (rentang 0.01-0.40)

Sama dengan threshold tersimpan?
Uncalibrated: True
Calibrated  : True

                Skenario  Recall_0  Recall_1  Recall_2  Macro_F1  G_Mean  Accuracy
Uncalibrated + best 0.39    0.6303    0.4251    0.5205    0.4158  0.5186    0.6112
  Calibrated + best 0.03    0.8297    0.3552    0.1455    0.3856  0.3500    0.7257


## 6. Kualitas Probabilitas

Calibration bertujuan membuat probabilitas lebih jujur, bukan menaikkan recall. Karena itu penilaian yang adil memakai **Log Loss** dan **Brier Score** (lebih kecil lebih baik), selain metrik klasifikasi di atas.

Baris terakhir membandingkan rata-rata probabilitas Prediabetes dengan proporsi Prediabetes yang sebenarnya pada data train.

In [7]:
def brier_multiclass(y_true, proba):
    onehot = np.eye(3)[y_true]
    return np.mean(np.sum((proba - onehot) ** 2, axis=1))


rows = []
for name, proba in [("Uncalibrated", proba_uncal), ("Calibrated", proba_cal)]:
    rows.append({
        "Skenario": name,
        "Log_Loss": log_loss(y, proba, labels=[0, 1, 2]),
        "Brier": brier_multiclass(y, proba),
        "Rata2_Prob_Kelas_1": proba[:, 1].mean(),
        "Proporsi_Kelas_1_Asli": (y == 1).mean(),
    })

print(pd.DataFrame(rows).round(4).to_string(index=False))

    Skenario  Log_Loss  Brier  Rata2_Prob_Kelas_1  Proporsi_Kelas_1_Asli
Uncalibrated    0.8715 0.4891              0.2919                 0.0183
  Calibrated    0.3946 0.2209              0.0185                 0.0183


## 7. Cara Membaca Hasil

- Jika **Log Loss dan Brier turun** setelah calibration tetapi **Recall Prediabetes dan G-Mean turun**, itu pola yang wajar. Meta-learner memakai `class_weight="balanced"` yang sengaja menaikkan probabilitas kelas minoritas. Platt Scaling lalu menyesuaikan probabilitas ke frekuensi aslinya, sehingga probabilitas Prediabetes mengecil dan lebih jarang melewati batas keputusan.
- Calibration memperbaiki kejujuran probabilitas, sedangkan tujuan TA ini (deteksi dini Prediabetes pada data tidak seimbang) menuntut recall kelas minoritas yang tinggi. Dua tujuan ini tidak selalu searah.
- Bagian 5 menunjukkan apakah perbedaan itu bertahan setelah threshold sama-sama disesuaikan. Bagian 3 dan 4 menunjukkan perbedaan sebelum penyesuaian.
- Jika calibration tidak membantu, hasil itu tetap sah dilaporkan sebagai temuan: calibration dicoba dan tidak dipilih karena menurunkan recall Prediabetes, sedangkan model final memakai threshold tuning tanpa calibration. Ini juga sesuai dengan `predict.py`, yang memakai `prediabetes_threshold.txt`.
- Catatan: threshold pada notebook ini dicari pada data yang sama dengan yang dievaluasi, sehingga angka kedua skenario sedikit optimistis. Perbandingannya tetap adil karena perlakuannya sama untuk keduanya.